# Exo nettoyage des données #

## Import ##

In [85]:
# Import des bibliotéque
import pandas as pd
import numpy as np

In [71]:
# Import des sources de données
data_operations = pd.read_csv("./sources/operations.csv")
data_operations.head()

,date_operation,libelle,montant,solde_avt_ope,categ
0,2023-03-31,DON XX XX XX XX XX XX XX,-1.44,1515.25,AUTRE
1,2023-04-03,CARTE XX XX RAPT XX,-24.00,1513.81,TRANSPORT
2,2023-04-03,CARTE XX XX RAPT XX,-73.00,1489.81,TRANSPORT
3,2023-04-03,VIREMENT XX XX XX XX XX XX XX XX XX XX XX XX,676.00,1416.81,AUTRE
4,2023-04-03,VIREMENT XX XX XX XX XX XX,4.80,2092.81,AUTRE


## Gestion des erreurs ##

* Erreur de type de variable

In [72]:
data_operations.dtypes

date_operation     object
libelle            object
montant           float64
solde_avt_ope     float64
categ              object
dtype: object

In [73]:
# Gestion des erreurs de type de daye
data_operations['date_operation'] = pd.to_datetime(data_operations['date_operation'], errors='coerce')

In [74]:
data_operations.dtypes

date_operation    datetime64[ns]
libelle                   object
montant                  float64
solde_avt_ope            float64
categ                     object
dtype: object

* Valeur manquantes

In [75]:
data_operations.isnull().sum()

date_operation    0
libelle           0
montant           2
solde_avt_ope     0
categ             1
dtype: int64

In [76]:
# pour afficher uniquement les variables qui ont des valeurs manquantes
nb_na = data_operations.isnull().sum()
nb_na[nb_na>0]

montant    2
categ      1
dtype: int64

In [77]:
data_operations.loc[data_operations['montant'].isnull(),:]

,date_operation,libelle,montant,solde_avt_ope,categ
107,2023-06-12,CARTE XX XX LES ANCIENS ROBINSON XX,NaN,4667.19,COURSES
269,2023-09-11,CARTE XX XX XX XX,NaN,3401.93,AUTRE


In [79]:
# On affiche les lignes avec des valeurs manquantes dans la colonne 'montant'
data_operations.loc[data_operations['montant'].isnull(),:]

# on stocke le df des valeurs manquantes dans un nouveau df
data_na = data_operations.loc[data_operations['montant'].isnull(),:]

# pour chaque ligne de mon df, on récupère les index (qui ne changent pas au travers du .loc)
for index in data_na.index:
    # calcul du montant à partir des soldes précédents et actuels
    data_operations.loc[index, 'montant'] = data_operations.loc[index+1, 'solde_avt_ope'] - data_operations.loc[index, 'solde_avt_ope']

data_operations.loc[data_operations['categ'].isnull(),:]

data_operations.loc[data_operations['libelle'] == 'PRELEVEMENT XX TELEPHONE XX XX', :]

data_operations.loc[data_operations['categ'].isnull(), 'categ'] = 'FACTURE TELEPHONE'

* Doublons

In [80]:
data_operations.loc[data_operations[['date_operation', 'libelle', 'montant', 'solde_avt_ope']].duplicated(keep=False),:]

,date_operation,libelle,montant,solde_avt_ope,categ
43,2023-04-25,CARTE XX XX LES ANCIENS ROBINSON XX,-32.67,3647.67,COURSES
44,2023-04-25,CARTE XX XX LES ANCIENS ROBINSON XX,-32.67,3647.67,COURSES


In [81]:
data_operations.drop_duplicates(subset=['date_operation', 'libelle', 'montant', 'solde_avt_ope'], inplace=True, ignore_index=True)

* Outlier

In [82]:
data_operations.describe()

,date_operation,montant,solde_avt_ope
count,308,308.000000,308.000000
mean,2023-07-05 10:59:13.246753280,-45.782013,3395.301071
min,2023-03-31 00:00:00,-15000.000000,1416.810000
25%,2023-05-21 06:00:00,-20.447500,3010.737500
50%,2023-07-05 12:00:00,-9.600000,3452.465000
75%,2023-08-21 00:00:00,-2.715000,3787.232500
max,2023-10-06 00:00:00,1071.600000,4709.310000
std,NaN,872.818105,667.109412


In [83]:
i = data_operations.loc[data_operations['montant']==-15000,:].index[0] # récupération de l'index de la transaction à -15000

data_operations.iloc[i-1:i+2,:] # on regarde la transaction précédente et la suivante

,date_operation,libelle,montant,solde_avt_ope,categ
197,2023-08-03,VIREMENT XX XX XX XX XX XX XX XX XX XX XX XX,676.00,3121.35,AUTRE
198,2023-08-03,CARTE XX XX XX XX,-15000.00,3797.35,AUTRE
199,2023-08-03,CARTE XX XX L'EPICERIE DEMBAS XX XX,-10.51,3782.96,AUTRE


In [84]:
data_operations.loc[data_operations['montant']==-15000, 'montant'] = -14.39